# Flatten the model

## Prune ResNet (works!!!)

In [1]:
import torch
import torch.nn as nn
import torch_pruning as tp
from torchvision import models
import random
import torch_pruning as tp
from ultralytics import YOLOv10

# Load a pre-trained ResNet model
model = models.resnet50(pretrained=True)

# Set the model to evaluation mode to prevent any unintended training behavior
#print(model)

# Function to prune the model
def prune_model(model, prune_percentage):
    # Create a dependency graph while the model is in evaluation mode
    DG = tp.DependencyGraph().build_dependency(model, example_inputs=torch.randn(1, 3, 224, 224))

    # Function to prune a single layer
    def prune_layer(layer, amount):
        if isinstance(layer, nn.Conv2d):
            # Generate a pruning plan for pruning output channels
            pruning_group = DG.get_pruning_group(layer, tp.prune_conv_out_channels, idxs=[0,1,2,3])
            # Execute the pruning plan
            pruning_group.prune()

    # Flatten the model to access each layer
    flattened_layers = [module for module in model.modules() if isinstance(module, nn.Conv2d)]

    # Apply pruning to each convolutional layer
    for i, layer in enumerate(flattened_layers):
        print(i, layer)
        amount = random.uniform(0, prune_percentage)
        prune_layer(layer, amount)
        print(layer)

# Apply pruning to the ResNet model with up to 50% pruning per layer
prune_percentage = 0.5
group_prune_model(model) #, prune_percentage)

print("pruning is done")
# Ensure the model remains in evaluation mode after pruning
model.eval()

# Print the pruned model to verify
#print(model)


/data/blanka/virtualenvs/yolov10_p12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/data/blanka/virtualenvs/yolov10_p12/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/data/blanka/virtualenvs/yolov10_p12/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


NameError: name 'group_prune_model' is not defined

# Prune YOLOv10 (works until layer 82)

In [1]:
%cd ..

/home/blanka/Multi-Domain-Pruning


In [7]:
import torch
import torch.nn as nn
import torch_pruning as tp
from torchvision import models
import random
import numpy as np
import torch_pruning as tp
from ultralytics import YOLOv10, YOLO
from ultralytics.nn.modules import Detect, ChannelAttention, SpatialAttention, C2f
import copy
import time
import gc

model = "yolov7" # yolov5, 7, 8, 10
device = "cuda"
example_inputs = torch.randn(1, 3, 640, 640).to(device)


if model == "yolov10":
    init_model = YOLOv10.from_pretrained('jameslahm/yolov10x').to(device)
    model = copy.deepcopy(init_model)
    model = model.model.train()  
elif model == "yolov8":
    init_model = YOLO('yolov8x.pt').to(device)
    model = copy.deepcopy(init_model) 
    model = model.model.train() 
    for name, param in model.model.named_parameters():
        param.requires_grad = True 
elif model == "yolov5":
    init_model = YOLO('yolov5su.pt').to(device)
    model = copy.deepcopy(init_model) 
    model = model.model.train() 
    for name, param in model.model.named_parameters():
        param.requires_grad = True 
elif model == "yolov7":
    init_model = torch.load('yolov7.pt', weights_only=False).to(device)
    print(init_model)




def prune_model(model):

    ignored_layers = []
    prunable_layers = []
    for name, module in model.named_modules():
        if "attn" in name and isinstance(m, (nn.modules.Conv2d, nn.modules.BatchNorm2d)):
            ignored_layers.append(m)
        # elif isinstance(module, C2f):
        #     for m in module.modules():
        #         if isinstance(m, nn.Conv2d):
        #             ignored_layers.append(m)
            

    DG = tp.DependencyGraph()
    DG.build_dependency(model, example_inputs=example_inputs) #ignored_layers=ignored_layers)     


    def _prune_layer(layer, idxs):

        if isinstance(layer, nn.Conv2d):
            pruning_group = DG.get_pruning_group(layer, tp.prune_conv_out_channels, idxs)
            DG.check_pruning_group(pruning_group)
            pruning_group.prune()

    prunable_layers = []
    for name, m in model.named_modules():
        if (isinstance(m, nn.Conv2d) and not ("attn" in name)):
            prunable_layers.append(m)

    
    for i, layer in enumerate(prunable_layers): 
        
        if layer not in list(DG.module2node.keys()):
            continue

        print(f"Pruning layer {i}: {layer}")
        start_time = time.time()

        if layer in ignored_layers: 
            continue
        
        n_channels_to_prune = int(0.8 * layer.out_channels)
        idxs=np.random.choice(range(layer.out_channels), n_channels_to_prune, replace=False)     
        #idxs = [1,2]    
        print(f"{len(idxs) = }")
        _prune_layer(layer, idxs)
        #print_model_weights(model)
        
        end_time = time.time()
        print(f"Layer {i} pruned: {layer}")
        print(f"Layer {i} pruned in {end_time - start_time:.2f} seconds")
        del layer
        gc.collect()
        model.zero_grad()
        init_model.model = copy.deepcopy(model)
        #print(model)
        #print(init_model)
        init_model.zero_grad()
        #print_model_weights(init_model)
        prec_metrics = init_model.val(data="config/data/kitti.yaml", batch=2, plots=None)

def group_prune_model(model):
    ignored_layers = []

    ignored_layers = []
    unwrapped_parameters = []
    imp = tp.importance.MagnitudeImportance(p=2)
    for m in model.modules():
        if isinstance(m, (Detect,)):
            ignored_layers.append(m)

    pruner = tp.pruner.MagnitudePruner(
            model,
            example_inputs,
            importance=imp,
            iterative_steps=1,
            pruning_ratio=0.5, # remove 50% channels, ResNet18 = {64, 128, 256, 512} => ResNet18_Half = {32, 64, 128, 256}
            ignored_layers=ignored_layers,
    )
    pruner.step()
    init_model.model = copy.deepcopy(model)
    #print(model)
    print(init_model)
    #prec_metrics = init_model.val(data="config/data/kitti.yaml", batch=2, plots=None)


prune_model(model)
model.eval()
model.zero_grad()
torch.save(model, 'model_pruned.pth')
model = torch.load('model_pruned.pth')
print(model)


ModuleNotFoundError: No module named 'models'

In [3]:
def print_model_weights(model):
    print("Layer Name".ljust(50), "Weight Shape")
    print("="*80)

    for name, param in model.named_parameters():
        if param.requires_grad:  # Only print trainable parameters
            print(name.ljust(50), str(tuple(param.shape)))

In [ ]:

def find_negative_output_channels(model):
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            if module.out_channels < 0:
                print(f"Negative output channels detected in layer: {name} ({module.out_channels})")
                return name  # Return the name of the problematic layer
        elif isinstance(module, nn.BatchNorm2d):
            if module.num_features < 0:
                print(f"Negative features detected in BatchNorm layer: {name} ({module.num_features})")
                return name  # Return the name of the problematic layer
    print("No negative output channels detected.")
    return None